# Cognitive Neuroscience Per-Layer Extraction and Brain-Score Evaluation

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


Cognitive neuroscience workflows usually need one representation per stimulus, per layer, then a benchmark score per representation. This notebook uses TorchLens' public Brain-Score bridge with a deterministic offline callable benchmark, the same fixture style used in tests so it stays network-free.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1605
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyVisionModel(nn.Module):
    """Small CNN for image-like stimulus activations."""

    def __init__(self) -> None:
        """Initialize convolutional and readout layers."""
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(4, 4, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.readout = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return class-like outputs from image stimuli."""
        features = self.features(x).mean(dim=(-2, -1))
        return self.readout(features)


def offline_brain_score_fixture(
    activation: torch.Tensor, *, layer: str, target: torch.Tensor
) -> float:
    """Score a layer by correlation with a deterministic neural target."""
    responses = activation.reshape(activation.shape[0], -1).mean(dim=1)
    centered_responses = responses - responses.mean()
    centered_target = target - target.mean()
    numerator = torch.dot(centered_responses, centered_target)
    denominator = torch.linalg.vector_norm(centered_responses) * torch.linalg.vector_norm(
        centered_target
    )
    return float(numerator / denominator.clamp_min(1e-8))

In [ ]:
model = TinyVisionModel().eval()
stimuli = torch.linspace(-1.0, 1.0, steps=6 * 1 * 8 * 8).reshape(6, 1, 8, 8)
neural_target = torch.tensor([-0.8, -0.4, -0.1, 0.2, 0.5, 0.9])
log = tl.log_forward_pass(model, stimuli, vis_opt="none", intervention_ready=True)
sites = [site.layer_label for site in log.find_sites(tl.func("relu"))]
print(sites)

In [ ]:
scores = tl.bridge.brain_score.per_layer(
    log, offline_brain_score_fixture, sites=sites, target=neural_target
)
for layer, score in scores.items():
    print(f"{layer:18s} Brain-Score fixture correlation {score: .3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(range(len(scores)), list(scores.values()), color="#4c78a8")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xticks(range(len(scores)))
ax.set_xticklabels(list(scores), rotation=25, ha="right")
ax.set_ylabel("offline score")
ax.set_title("Per-layer Brain-Score-style evaluation")
plt.tight_layout()
plt.show()

When the real Brain-Score benchmark is available, keep the same TorchLens extraction pattern and replace the callable fixture with the benchmark adapter. The important separation is that TorchLens owns layer capture and site selection; the downstream benchmark owns the scoring semantics.
